# OSFT Continual Learning: Ministral 3 3B on Medical Flashcards

This notebook demonstrates **Orthogonal Subspace Fine-Tuning (OSFT)** of `mistralai/Ministral-3-3B-Instruct-2512` on a medical Q&A dataset using Training Hub.

**What is OSFT?** OSFT constrains weight updates to low-rank subspaces orthogonal to the model's critical knowledge subspace. This enables domain adaptation (adding medical knowledge) **without catastrophic forgetting** — the model retains its general reasoning and language abilities.

**Goal:** Teach the model to produce concise medical answers while preserving its ability to handle general-purpose questions.

**Dataset:** [medalpaca/medical_meadow_medical_flashcards](https://huggingface.co/datasets/medalpaca/medical_meadow_medical_flashcards) — 33,955 medical Q&A pairs covering anatomy, pharmacology, pathology, and clinical medicine.

**Model:** Ministral 3 3B is a compact instruction-tuned model from Mistral AI. Training Hub extracts the CausalLM text backbone from its VLM wrapper for efficient text-only fine-tuning. Note that **Liger kernels are not yet supported** for the Ministral 3 architecture.

**Hardware requirements:**
- Minimum: 2× GPUs with 40GB+ VRAM (e.g., A100-40GB, L40S)
- Recommended: 8× A100-80GB for faster training

**Prerequisites:**
```bash
pip install -e . && pip install -e .[cuda] --no-build-isolation
```

**Learning objectives:**
- Fine-tune a model for medical domain Q&A with OSFT
- Verify the trained model produces factual medical answers
- Test knowledge retention to confirm no catastrophic forgetting

## Setup and Imports

In [ ]:
from training_hub import osft

import glob
import json
import os
import time
import logging
import sys
from io import StringIO
from contextlib import redirect_stdout, redirect_stderr

## Logging Configuration

In [ ]:
logging.basicConfig(level=logging.WARNING, format='%(name)s - %(levelname)s - %(message)s')
for logger_name in ['transformers', 'datasets', 'torch', 'accelerate']:
    logging.getLogger(logger_name).setLevel(logging.ERROR)

## Utility Functions

In [ ]:
def find_most_recent_checkpoint(output_dir):
    """Find the most recent HF-format checkpoint in the training output directory."""
    # OSFT checkpoints use a float suffix (e.g., samples_9000.0)
    checkpoint_dirs = glob.glob(os.path.join(output_dir, "hf_format", "samples_*.0"))
    if not checkpoint_dirs:
        return None
    return max(checkpoint_dirs, key=os.path.getctime)

## Load and Prepare the Dataset

The [medical_meadow_medical_flashcards](https://huggingface.co/datasets/medalpaca/medical_meadow_medical_flashcards) dataset contains concise medical Q&A pairs. We convert them to the JSONL **messages** format expected by Training Hub.

| Field | Example |
|-------|--------|
| `input` | *What is the mechanism of action of ACE inhibitors?* |
| `output` | *ACE inhibitors block the conversion of angiotensin I to angiotensin II, reducing vasoconstriction and aldosterone secretion.* |

In [ ]:
from datasets import load_dataset

ds = load_dataset("medalpaca/medical_meadow_medical_flashcards", split="train")
print(f"Full dataset: {len(ds):,} samples")

# Use a subset for this demo
SUBSET_SIZE = 3000
ds_subset = ds.shuffle(seed=42).select(range(SUBSET_SIZE))

# Convert to JSONL messages format
DATA_PATH = "./ministral_osft_output/medical_flashcards.jsonl"
os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)

with open(DATA_PATH, "w") as f:
    for row in ds_subset:
        messages = [
            {"role": "user", "content": row["input"]},
            {"role": "assistant", "content": row["output"]},
        ]
        f.write(json.dumps({"messages": messages}) + "\n")

print(f"Wrote {SUBSET_SIZE} samples to {DATA_PATH}")

# Show a few examples
for i, row in enumerate(ds_subset.select(range(3))):
    print(f"\n--- Sample {i} ---")
    print(f"  Q: {row['input'][:120]}")
    print(f"  A: {row['output'][:120]}")

## Configuration

Adjust `NUM_GPUS` and `MAX_TOKENS_PER_GPU` to match your hardware. The `UNFREEZE_RANK_RATIO` controls what fraction of each weight matrix's rank is trainable — higher values give more capacity but increase memory usage.

**Note:** `USE_LIGER` is set to `False` because Liger kernels do not yet support the Ministral 3 architecture.

In [ ]:
# ── Model + Data Paths ──
MODEL_PATH = "mistralai/Ministral-3-3B-Instruct-2512"
CKPT_OUTPUT_DIR = "./ministral_osft_output/checkpoints"

# ── OSFT-Specific ──
UNFREEZE_RANK_RATIO = 0.25
USE_LIGER = False  # Liger not supported for Ministral 3

# ── Training Hyperparameters ──
NUM_EPOCHS = 3
EFFECTIVE_BATCH_SIZE = 32
LEARNING_RATE = 5e-6
MAX_SEQ_LEN = 4096
WARMUP_STEPS = 0
LR_SCHEDULER = "cosine"
SEED = 42

# ── Performance ──
MAX_TOKENS_PER_GPU = 8192

# ── Checkpointing ──
CHECKPOINT_AT_EPOCH = True
SAVE_FINAL_CHECKPOINT = True

# ── Torchrun / Multi-GPU ──
NUM_GPUS = 8
NNODES = 1
NODE_RANK = 0
RDZV_ID = 301
RDZV_ENDPOINT = "localhost:29501"

print(f"Model:              {MODEL_PATH}")
print(f"Data:               {DATA_PATH} ({SUBSET_SIZE} samples)")
print(f"Output:             {CKPT_OUTPUT_DIR}")
print(f"Epochs:             {NUM_EPOCHS}")
print(f"Effective batch:    {EFFECTIVE_BATCH_SIZE}")
print(f"Learning rate:      {LEARNING_RATE}")
print(f"Unfreeze ratio:     {UNFREEZE_RANK_RATIO}")
print(f"Liger kernels:      {USE_LIGER}")
print(f"GPUs:               {NUM_GPUS}")
print(f"Max tokens/GPU:     {MAX_TOKENS_PER_GPU:,}")

## Training with OSFT

OSFT performs an SVD decomposition of each target weight matrix before training to identify the orthogonal subspace. This adds startup time compared to SFT, but preserves the model's existing capabilities.

In [ ]:
start_time = time.time()

captured_out = StringIO()
captured_err = StringIO()

try:
    with redirect_stdout(captured_out), redirect_stderr(captured_err):
        osft(
            model_path=MODEL_PATH,
            data_path=DATA_PATH,
            ckpt_output_dir=CKPT_OUTPUT_DIR,
            unfreeze_rank_ratio=UNFREEZE_RANK_RATIO,
            num_epochs=NUM_EPOCHS,
            effective_batch_size=EFFECTIVE_BATCH_SIZE,
            learning_rate=LEARNING_RATE,
            max_seq_len=MAX_SEQ_LEN,
            max_tokens_per_gpu=MAX_TOKENS_PER_GPU,
            data_output_dir=os.path.join(CKPT_OUTPUT_DIR, "_data"),
            warmup_steps=WARMUP_STEPS,
            use_liger=USE_LIGER,
            seed=SEED,
            lr_scheduler=LR_SCHEDULER,
            checkpoint_at_epoch=CHECKPOINT_AT_EPOCH,
            save_final_checkpoint=SAVE_FINAL_CHECKPOINT,
            nproc_per_node=NUM_GPUS,
            nnodes=NNODES,
            node_rank=NODE_RANK,
            rdzv_id=RDZV_ID,
            rdzv_endpoint=RDZV_ENDPOINT,
        )

    elapsed = time.time() - start_time
    ckpt = find_most_recent_checkpoint(CKPT_OUTPUT_DIR)

    print(f"Training completed in {elapsed/60:.1f} minutes")
    print(f"Checkpoint saved to: {ckpt}")

except Exception as e:
    elapsed = time.time() - start_time
    print(f"Training failed after {elapsed/60:.1f} minutes")
    print(f"Error: {e}")
    print("\nStderr output:")
    print(captured_err.getvalue()[-2000:])

## Testing the Trained Model: Medical Questions

Let's verify the OSFT-trained model produces concise, factual medical answers.

In [ ]:
import torch
from transformers import Ministral3ForCausalLM, AutoTokenizer

ckpt_path = find_most_recent_checkpoint(CKPT_OUTPUT_DIR)
print(f"Loading checkpoint: {ckpt_path}")

tokenizer = AutoTokenizer.from_pretrained(ckpt_path)
model = Ministral3ForCausalLM.from_pretrained(
    ckpt_path,
    dtype=torch.bfloat16,
    device_map="cuda:0",
)
model.eval()


def generate_answer(question, max_tokens=256):
    messages = [{"role": "user", "content": question}]
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=True,
            temperature=0.6,
            top_p=0.95,
        )
    return tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()


medical_questions = [
    "What are the classic signs and symptoms of diabetic ketoacidosis?",
    "What is the mechanism of action of ACE inhibitors in treating hypertension?",
    "What is the most common cause of iron deficiency anemia in adults?",
    "What are the differences between Type 1 and Type 2 diabetes mellitus?",
]

print("=" * 70)
print("  Medical Domain Test")
print("=" * 70)
for i, q in enumerate(medical_questions):
    answer = generate_answer(q)
    print(f"\nQ{i+1}: {q}")
    print(f"A{i+1}: {answer}")
    print("-" * 70)

## Testing Knowledge Retention: General Questions

The key advantage of OSFT over standard SFT is **knowledge retention**. Let's verify the model still handles general-purpose questions correctly — something it knew *before* medical fine-tuning.

**Expected behavior:** The model should provide a coherent, accurate answer about general science, demonstrating that medical fine-tuning did not cause catastrophic forgetting of its pre-training knowledge.

In [ ]:
general_questions = [
    "Explain why the sky appears blue during the day.",
    "What is the difference between a compiled and an interpreted programming language?",
    "Summarize the main causes of World War I.",
]

print("=" * 70)
print("  Knowledge Retention Test (non-medical questions)")
print("=" * 70)
for i, q in enumerate(general_questions):
    answer = generate_answer(q)
    print(f"\nQ{i+1}: {q}")
    print(f"A{i+1}: {answer}")
    print("-" * 70)

print("\nIf the model produced coherent, accurate answers to the non-medical")
print("questions above, OSFT successfully preserved its general knowledge.")

## Summary

In this notebook we:

1. **Loaded** the medical flashcards dataset and converted it to JSONL messages format
2. **Fine-tuned** Ministral 3 3B using OSFT on 3,000 medical Q&A pairs
3. **Verified** the trained model produces concise, factual medical answers
4. **Tested knowledge retention** to confirm no catastrophic forgetting of general knowledge

**How OSFT differs from SFT:**
- OSFT constrains updates to orthogonal subspaces, preserving existing capabilities
- SFT may overwrite general knowledge when adapting to a specialized domain
- OSFT is ideal for **continual learning** — adding domain expertise without losing what the model already knows

**Next steps:**
- Compare with the SFT notebook (`runnable_ministral_sft.ipynb`) to see the difference in knowledge retention
- Try LoRA (`runnable_ministral_lora.ipynb`) for parameter-efficient fine-tuning on a single GPU
- Experiment with `unfreeze_rank_ratio` — higher values (0.3–0.5) give more learning capacity at the cost of more forgetting
- Scale up to the full 33,955-sample dataset for stronger domain adaptation